# 14 — Authoring geometry natively: the shapes route

**Who this is for.** Anyone who wants to build an instrument in `ion_gym`
and has been asking "where do I draw the electrodes?"

**The short answer: in the spec.** Geometry is *declared* — a list of
electrodes, each a list of shapes with dimensions in millimetres — and the
solver rasterizes that declaration onto its own grid. There is no separate
geometry file format, no external editor, and no import step. This notebook
walks the whole vocabulary: rectangles, polygons, extrusion into 3-D, and
the ordering rules that keep a build fast.


## 0. Paths

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (charter: stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
from pathlib import Path
from ion_gym.io.paths import repo_root
ROOT = Path(repo_root())


## 1. Parameters

One cell, every number named with its units and reason.

In [ ]:
import numpy as np

# ---- the worked example: a three-electrode stack ------------------------
TUBE_L_MM = 20.0          # electrode axial length [mm]
BORE_R_MM = 6.0           # inner radius of the outer electrodes [mm]
BORE_R_MID_MM = 5.0       # inner radius of the middle electrode [mm]
WALL_T_MM = 1.5           # wall thickness [mm]
GAP_MM = 4.0              # axial gap between electrodes [mm]
V_MID = 48.0              # middle-electrode bias [V] — MEASURED sweep
                          # argmin (seed 0): mean |r_end| at
                          # the exit plane 40 V -> 1.89 mm, 47 -> 1.92,
                          # 48 -> 0.81 (all 6 detected), 48.5 -> 3.66;
                          # above 48.8 the crossover overfocuses into the
                          # middle bore (ion strikes), 49.5 reflects.
                          # Decel mode: V_mid just under the 50 eV beam.

# ---- domain and rasterization -------------------------------------------
PAD_MM = 10.0             # drift space at each end [mm]
DOMAIN_R_MM = 12.0        # radial extent [mm]
PITCH_MM = 0.2            # raster pitch [mm]: every dimension is snapped
                          # to this grid, so features smaller than ~2*PITCH
                          # are not resolved

# ---- the 3-D demonstration ----------------------------------------------
DEPTH_MM = 8.0            # z extent of the 3-D box [mm]
PITCH_3D_MM = 0.5         # coarser: a 3-D solve costs O(n^3)

# ---- ions (a geometry notebook still shows the geometry WORKING) --------
MZ, KE_EV, N_IONS = 100.0, 50.0, 6
print(f"stack length {3*TUBE_L_MM + 2*GAP_MM:g} mm at pitch {PITCH_MM:g} mm")

## 2. The vocabulary

A `GeometrySpec` carries the domain (`width_mm` x `height_mm`, optionally
`depth_mm`), the raster `mm_per_gu`, a `SymmetrySpec`, and the electrodes.
Each `ElectrodeSpec` has a name, a bias, and a list of `ShapeSpec`s. Three
shape types cover almost everything:

| type | params | notes |
|---|---|---|
| `rect` | `x_mm`, `y_mm`, `width_mm`, `height_mm` | lower-left anchored |
| `polygon` | `points_mm` — list of `[x, y]` | for curves, sample them |
| `ellipse` | centre and radii | round bodies |

In an **r-z** spec (`coords="rz"`) the second coordinate is *radius*, not y:
a `rect` at `y_mm = 6` is a tube wall at r = 6 mm, revolved by the solver.
A `polygon` is how curved electrodes are built — sample the analytic curve
and hand over the points, which is exactly what the shipped Paul trap deck
does with its hyperbolae.

In [ ]:
from ion_gym.io.sim_spec import (SimSpec, GeometrySpec, ElectrodeSpec,
                                 ShapeSpec, SourceSpec, IntegrationSpec,
                                 BoundsSpec, CollisionSpec, SymmetrySpec)

def tube(name, x0_mm, r_in_mm, dc):
    """One cylindrical electrode: a rect in r-z IS a tube."""
    return ElectrodeSpec(name=name, dc=dc, rf_groups=[], shapes=[
        ShapeSpec(type="rect", params={
            "x_mm": x0_mm, "y_mm": r_in_mm,
            "width_mm": TUBE_L_MM, "height_mm": WALL_T_MM})])

STACK_L = 3 * TUBE_L_MM + 2 * GAP_MM
starts = [PAD_MM + k * (TUBE_L_MM + GAP_MM) for k in range(3)]
geom = GeometrySpec(
    width_mm=STACK_L + 2 * PAD_MM, height_mm=DOMAIN_R_MM,
    mm_per_gu=PITCH_MM, symmetry=SymmetrySpec(coords="rz"),
    electrodes=[tube("entrance", starts[0], BORE_R_MM, 0.0),
                tube("middle", starts[1], BORE_R_MID_MM, V_MID),
                tube("exit", starts[2], BORE_R_MM, 0.0)])
spec = SimSpec(
    name="14 shapes demo (r-z)", geometry=geom,
    source=SourceSpec(n_ions=N_IONS, distribution="disc", r_mm=2.0, axis="x",
                      x0_mm=1.0, y0_mm=0.0, mz_list=[MZ], charge=1,
                      ke_lo=KE_EV, ke_hi=KE_EV, direction=(1.0, 0.0, 0.0),
                      temperature_k=0.0, seed=0),
    integration=IntegrationSpec(t_max_us=20.0, dt_ns=2.0, rec_every=4),
    bounds=BoundsSpec(x_max_on=True, x_max=STACK_L + 2 * PAD_MM),
    collisions=CollisionSpec(enabled=False))
err = spec.validate()
if err:
    raise RuntimeError(f"spec refused: {err}")
print("declared:", [e.name for e in spec.geometry.electrodes])

## 3. Ordering: build, then solve, then set voltages

The order is not stylistic. Geometry defines the grid; the solve produces
one basis per electrode; **voltages then reweight those cached bases**,
which is over 200x faster than re-solving. So: build the geometry once,
solve once, and change voltages as often as you like. Rebuilding the spec
from scratch to change a voltage throws away the expensive part.

In [ ]:
from ion_gym.physics.sim_build import build_run
model, fly, cols, births = build_run(spec)
print(f"solved: grid {np.asarray(model.A).shape}, "
      f"{len(spec.geometry.electrodes)} bases cached")

## 4. The geometry, rendered from the solver's own mask

Never draw geometry twice. The figure below is not a re-drawing of the
numbers above — it is the rasterized mask the solver actually used, with
the ions exactly as flown. If the picture and the solve ever disagreed,
the picture would be the one lying.

In [ ]:
from ion_gym.viz.viz_core import scene_from_simspec, render_mpl
trajs, fates = [], []
for i in range(births.shape[0]):
    tr, st = fly(i)
    trajs.append(tr); fates.append(st["kind"])
sc = scene_from_simspec(spec, model, field="phi", trajs=trajs, fates=fates,
    title=(f"Shapes demo (r-z): three tubes, bores {BORE_R_MM:g}/"
           f"{BORE_R_MID_MM:g}/{BORE_R_MM:g} mm, gap {GAP_MM:g} mm | "
           f"V_mid = {V_MID:g} V | m/z {MZ:g} at {KE_EV:g} eV, "
           f"{N_IONS} ions | pitch {PITCH_MM:g} mm"))
render_mpl(sc, layout="column")

## 5. The whole vocabulary, per route

The planar/r-z rasterizer speaks exactly three primitives — `rect`,
`ellipse`, `polygon` — plus `cutout` (a shape minus its first child),
and the 3-D shapes route extrudes the same three along a declared axis
range. This section shows **every primitive, on every route**:
construction first (the declaration you would write), then the geometry
the *solver* rasterized — never a redrawing — so what you see is what a
flight would feel. Same sampler layout on each route, so the routes can
be compared shape for shape.

In [ ]:
def hyperbolic_ring(r0_mm, z_half_mm, zc_mm, r_out_mm, n=181):
    """Ring of a hyperbolic trap: r(z) = sqrt(r0^2 + 2 (z-zc)^2)."""
    z = np.linspace(zc_mm - z_half_mm, zc_mm + z_half_mm, n)
    r = np.sqrt(r0_mm ** 2 + 2.0 * (z - zc_mm) ** 2)
    pts = list(zip(z.tolist(), r.tolist()))
    return pts + [(zc_mm + z_half_mm, r_out_mm), (zc_mm - z_half_mm, r_out_mm)]

# --- the sampler: one electrode per primitive, same layout every route ---
# Construction is the point of this cell: each electrode below IS the
# declaration a deck would carry.
def sampler_electrodes():
    return [
        ElectrodeSpec(name="rect_bar", dc=10.0, rf_groups=[], shapes=[
            ShapeSpec(type="rect", params={
                "x_mm": 2.0, "y_mm": 6.0, "width_mm": 8.0,
                "height_mm": 2.0})]),
        ElectrodeSpec(name="ellipse_ring", dc=20.0, rf_groups=[], shapes=[
            ShapeSpec(type="ellipse", params={
                "cx_mm": 18.0, "cy_mm": 7.0, "rx_mm": 4.0,
                "ry_mm": 2.0})]),
        ElectrodeSpec(name="poly_hyperbola", dc=30.0, rf_groups=[], shapes=[
            ShapeSpec(type="polygon", params={
                "points_mm": hyperbolic_ring(3.0, 4.0, 30.0, 9.0, n=121)})]),
    ]

def sampler_spec(coords, name):
    geom = GeometrySpec(width_mm=36.0, height_mm=12.0, mm_per_gu=PITCH_MM,
                        symmetry=SymmetrySpec(coords=coords),
                        electrodes=sampler_electrodes())
    return SimSpec(name=name, geometry=geom,
                   source=SourceSpec(n_ions=1, distribution="point",
                                     x0_mm=0.5, y0_mm=0.5, mz_list=[MZ]),
                   integration=IntegrationSpec(t_max_us=0.1, dt_ns=5.0),
                   bounds=BoundsSpec(), collisions=CollisionSpec(enabled=False))

from IPython.display import Image as _PNG, display as _display
import io as _io, matplotlib.pyplot as _plt

def show_scene(sc, height=520):
    fig = render_mpl(sc, views=["xy"]) if len(sc.fields) <= 1 else render_mpl(sc)
    buf = _io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    _display(_PNG(buf.getvalue(), height=height))
    _plt.close(fig)

# r-z: the SAME declarations read as bodies of revolution (y = radius)
spec_rz = sampler_spec("rz", "vocabulary sampler (r-z: revolved)")
m_rz, _f, _c, _b = build_run(spec_rz)
show_scene(scene_from_simspec(spec_rz, m_rz, field="phi",
    title=f"Shape vocabulary, r-z route (revolved about y = 0): rect bar, "
          f"ellipse ring, hyperbolic polygon | 10/20/30 V | pitch {PITCH_MM:g} mm"))

# planar: identical declarations, now a z-invariant cross-section
spec_pl = sampler_spec("xyz", "vocabulary sampler (planar cross-section)")
m_pl, _f, _c, _b = build_run(spec_pl)
show_scene(scene_from_simspec(spec_pl, m_pl, field="phi",
    title=f"Shape vocabulary, planar route (z-invariant cross-section): "
          f"same three declarations | 10/20/30 V | pitch {PITCH_MM:g} mm"))


## 6. Extrusion: the same vocabulary in 3-D

An `extrude` descriptor on any 2-D primitive turns the cross-section
into a 3-D solid over a declared axis range — the same three
declarations again, now with finite depth, solved on the 3-D route and
shown **multi-axis** (a 3-D instrument is never a single projection).

In [ ]:
def extruded(name, dc, shape_type, params, lo_mm, hi_mm):
    p = dict(params)
    p["extrude"] = {"axis": "z", "lo_mm": lo_mm, "hi_mm": hi_mm}
    return ElectrodeSpec(name=name, dc=dc, rf_groups=[],
                         shapes=[ShapeSpec(type=shape_type, params=p)])

geom3 = GeometrySpec(
    width_mm=36.0, height_mm=12.0, depth_mm=DEPTH_MM,
    mm_per_gu=PITCH_3D_MM, symmetry=SymmetrySpec(coords="xyz"),
    electrodes=[
        extruded("rect_slab", 10.0, "rect",
                 {"x_mm": 2.0, "y_mm": 6.0, "width_mm": 8.0,
                  "height_mm": 2.0}, 1.0, DEPTH_MM - 1.0),
        extruded("ellipse_rod", 20.0, "ellipse",
                 {"cx_mm": 18.0, "cy_mm": 7.0, "rx_mm": 4.0,
                  "ry_mm": 2.0}, 2.0, DEPTH_MM - 2.0),
        extruded("poly_prism", 30.0, "polygon",
                 {"points_mm": hyperbolic_ring(3.0, 4.0, 30.0, 9.0, n=61)},
                 1.0, DEPTH_MM - 1.0),
    ])
spec3 = SimSpec(name="14 shapes demo (3-D, extruded vocabulary)",
    geometry=geom3,
    source=SourceSpec(n_ions=1, distribution="point", x0_mm=0.5,
                      y0_mm=0.5, z0_mm=DEPTH_MM/2, mz_list=[MZ]),
    integration=IntegrationSpec(t_max_us=0.1, dt_ns=5.0, rec_every=20),
    bounds=BoundsSpec(), collisions=CollisionSpec(enabled=False))
m3, _f3, _c3, _b3 = build_run(spec3)
print(f"3-D grid {np.asarray(m3.A).shape} at pitch {PITCH_3D_MM:g} mm")
# multi-axis by default: render_mpl on a 3-D scene draws xy, xz, yz
show_scene(scene_from_simspec(spec3, m3, field="phi",
    title=f"Shape vocabulary, 3-D route (extruded): rect slab, ellipse rod, "
          f"hyperbolic prism | 10/20/30 V | pitch {PITCH_3D_MM:g} mm"),
    height=900)


## 7. Read-out — what this notebook established

**Established, by building and solving in this notebook:** every geometry
`ion_gym` flies is declared in the spec — `rect` for tubes and plates,
`polygon` for curved surfaces sampled finer than the raster pitch, and an
`extrude` descriptor to lift any planar shape into 3-D. The three-tube
stack solved, flew its ions, and rendered *from the solver's own mask*; the
3-D box built on the same vocabulary at a coarser pitch.

**What would have falsified it:** if the rendered mask had disagreed with
the declared dimensions, or if the polygon spacing check had reported
points coarser than the grid while the solve still claimed the curve, the
declaration would not be the geometry — and the whole premise of the route
would be wrong.

**Standing warnings.** `PITCH_MM` is the resolution limit: features
narrower than about two pitches are not resolved, so check the mask rather
than trusting the numbers you typed. And keep the ordering — build, solve,
*then* set voltages — because a voltage change reweights cached bases while
a geometry change forces a fresh solve.